In [0]:
%pip install git+https://github.com/jawong3/Databricks_Demos_and_Utilities.git@main
%restart_python

In [0]:
import time
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from dbx_utils.benchmark import *

N = 25_000_000
UNIQUE_ID = 25
SKEW_ID_AMOUNT = 0.5
SKEW_DIM_AMOUNT = 0.5

SHUFFLE_PARTS = 16

spark.conf.set("spark.databricks.io.cache.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))

In [0]:
@track_partitions
def create_event_transactions():
    return (
        # Purposefully create the table using 1000 partitions
        spark.range(0, N, 1, 1000)
        .withColumn("customer_id",
            f.when(f.rand(1) < SKEW_ID_AMOUNT, f.lit(0)).otherwise((f.col("id") % UNIQUE_ID) + 1))
        .withColumn("amount", (f.rand(2) * 100).cast("double"))
        .withColumn("event_date", f.date_add(f.lit("2026-01-01"), (f.col("id") % 365).cast("int")))
        .withColumn("quantity", (f.rand(3) * 50).cast("int"))
        .withColumn("discount", (f.rand(4) * 0.5).cast("double"))
        .withColumn("is_online", (f.rand(5) > 0.5).cast("boolean"))
    )

events = create_event_transactions()

customers = events.select("customer_id").distinct()

# Create a dimension table where the majority of customers share the same dimension value
dim_region = (customers.withColumn("region_id", (f.rand(1) * 10).cast("int")))
dim_region.write.format("delta").mode("overwrite").saveAsTable("dim_region")

In [0]:
# Filter the data such that partitions are either heavily reduced or untouched
def heavy_narrow_filter(df):
    return df.filter((f.col("id") < N * 0.05) | (f.rand(1) > 0.975))

# When a dataframe is written onto blob, each partition is written as its own file
# assuming no other optimizations were provided. Our example uses a narrow transformation
# such that the partitions never shuffles, thus the distribution of data soley depends on the filter.
# AQE never optimizes this either as it only coalesces shuffle partitions
@time_runs
@track_stages
def write_table(df, tableName, mode='overwrite'):
    df.write.format("delta").mode(mode).saveAsTable(tableName)

@time_runs
@track_stages
def write_partition_table(df, tableName, mode='overwrite', partitionKey="customer_id"):
    df.write.partitionBy(partitionKey).format("delta").mode(mode).saveAsTable(tableName)

random_filtered = heavy_narrow_filter(events)

In [0]:
# Should see the non coalseced table write the slowest as it has to
# scan through all 1000 partitions even if they are empty or near empty.
# The same will be true on read as there will be 1000 files to read
write_table(random_filtered, "non_coalesced_write")
print(f"Task Chain: {write_table.stages[-1]["task_chain"]}")
print(f"Total Task: {write_table.stages[-1]["total_tasks"]}")

# Coalseced should be considerably faster as it only has to scan through 20 partitions
write_table(random_filtered.coalesce(20), "coalesced_write")
print(f"Task Chain: {write_table.stages[-1]["task_chain"]}")
print(f"Total Task: {write_table.stages[-1]["total_tasks"]}")

# Repartitioning + partitioning performs slower than coalesce as it performs a wide transformation
# to group records with similar keys together. As a trade-off we achieve better read performance
# as it can use partition pruning
#
# Repartition also has another common use case: Per-partition before specific functions
# such as applyInPandas or foreachPartition to group co-related records together in tasks.
write_partition_table(random_filtered.repartition("customer_id"), "partition_write")
print(f"Task Chain: {write_partition_table.stages[0]["task_chain"]}")
print(f"Total Task: {write_partition_table.stages[0]["total_tasks"]}")

In [0]:
def get_table_file_stats(table):
    per_file = (spark.read.table(table)
        .select(
            f.col("_metadata.file_path").alias("file"),
            f.col("_metadata.file_size").alias("size_bytes"))
        .groupBy("file", "size_bytes")
        .agg(f.count("*").alias("row_count")))

    return (per_file
        .agg(
            f.count("*").alias("total_files"),
            f.round(f.avg("row_count"), 1).alias("avg_rows_per_file"),
            f.min("row_count").alias("min_rows"),
            f.max("row_count").alias("max_rows"))
        .withColumn("table", f.lit(table)))

tables = ["non_coalesced_write", "coalesced_write", "partition_write"]

summary = None
for t in tables:
    stats = get_table_file_stats(t)
    summary = stats if summary is None else summary.unionByName(stats)

(summary
    .select("table", "total_files", "avg_rows_per_file", "min_rows", "max_rows")
    .show(truncate=False))


In [0]:
@time_runs
@track_stages
def read_table(df): return spark.read.table(df).filter(f.col("customer_id") == 1)

df = read_table("non_coalesced_write")
print(f"Task Chain: {read_table.stages[-1]["task_chain"]}")
print(f"Total Task: {read_table.stages[-1]["total_tasks"]}")

df = read_table("coalesced_write")
print(f"Task Chain: {read_table.stages[-1]["task_chain"]}")
print(f"Total Task: {read_table.stages[-1]["total_tasks"]}")

df = read_table("partition_write")
print(f"Task Chain: {read_table.stages[-1]["task_chain"]}")
print(f"Total Task: {read_table.stages[-1]["total_tasks"]}")

In [0]:
(events
    .groupBy("customer_id")
    .count()
    .withColumn("table_proportion", f.concat(f.round(f.col("count")/N, 2).cast("string"), f.lit('%')))
    .orderBy("customer_id")
    .limit(5)
).show(truncate=False)

In [0]:
# TODO: UNSALTED TRANSFORMATION STILL OUTPERFORMS SALTED TRANSFORMATION WHERE IT SHOULDNT

SALT_BUCKETS = 25

dim = spark.table("dim_region")

@time_runs(runs=1)
@track_stages
def unsalted_join():
    return (events
        .join(dim, "customer_id")
        .groupBy("region_id")
        .agg(
            f.count("*").alias("record_count"),
            f.sum("amount").alias("total_amount"),
            f.sum("quantity").alias("total_quantity"))
        .withColumn("avg_amount", f.col("total_amount") / f.col("record_count")))

@time_runs(runs=1)
@track_stages
def salted_join():
    HOT = [0]
    salted_events = (events
        .withColumn("salt",
            f.when(f.col("customer_id").isin(HOT),
                f.floor(f.rand() * SALT_BUCKETS).cast("int"))
            .otherwise(f.lit(0))))

    hot_dim = (dim
        .filter(f.col("customer_id").isin(HOT))
        .withColumn("salt", f.explode(f.sequence(f.lit(0), f.lit(SALT_BUCKETS - 1)))))
    
    rest_dim = (dim
        .filter(~f.col("customer_id").isin(HOT))
        .withColumn("salt", f.lit(0)))

    salted_dim = hot_dim.unionByName(rest_dim)

    return (salted_events
        .join(salted_dim, ["customer_id", "salt"])
        .groupBy("region_id")
        .agg(
            f.count("*").alias("record_count"),
            f.sum("amount").alias("total_amount"),
            f.sum("quantity").alias("total_quantity"))
        .withColumn("avg_amount", f.col("total_amount") / f.col("record_count")))


unsalted_join().orderBy("region_id").limit(5).show(truncate=False)
print(f"Task Chain: {unsalted_join.stages[-1]["task_chain"]}")
print(f"Total Task: {unsalted_join.stages[-1]["total_tasks"]}")
salted_join().orderBy("region_id").limit(5).show(truncate=False)
print(f"Task Chain: {salted_join.stages[-1]["task_chain"]}")
print(f"Total Task: {salted_join.stages[-1]["total_tasks"]}")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.coalesced_write")
spark.sql(f"DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.non_coalesced_write")
spark.sql(f"DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.partition_write")
spark.sql(f"DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.dim_region")
spark.sql(f"DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.non_rep_partition_write")